# VLM-Anomaly — Full MVTec Sweep · Claude Opus 4.7

**Model:** `claude-opus-4-7`  
**Strategy:** 10 images per API call (batched) to maximise throughput  
**Free-tier Opus limits:** 50 RPM · 500K input TPM · 80K output TPM  

| Scope | Images | Est. cost |
|---|---|---|
| 1 category (smoke) | ~83 | ~$2.70 |
| Full sweep (15 cat) | ~1,245 | ~$26–32 |

> **$5 credit covers ~2 categories.** Add more credit before running the full sweep.

In [1]:
# ── Cell 1: Setup paths & sys.path ─────────────────────────────────────────
import sys
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT   = Path().resolve().parent
SRC_DIR     = REPO_ROOT / 'src'
PROMPTS_DIR = REPO_ROOT / 'prompts'
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert SRC_DIR.exists(),     f'src/ not found at {SRC_DIR}'
assert PROMPTS_DIR.exists(), f'prompts/ not found at {PROMPTS_DIR}'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(REPO_ROOT / '.env')

import vlm_anomaly
print(f'vlm_anomaly {vlm_anomaly.__version__} ready')
print(f'Results  : {RESULTS_DIR}')

vlm_anomaly 0.1.0 ready
Results  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/results


In [2]:
# ── Cell 2: Verify API key ───────────────────────────────────────────────────
import os

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
assert api_key, 'ANTHROPIC_API_KEY not set — add it to your .env file'
print(f'ANTHROPIC_API_KEY: {api_key[:12]}...{api_key[-4:]}')

ANTHROPIC_API_KEY: sk-ant-api03...SAAA


In [3]:
# ── Cell 3: Find MVTec dataset ──────────────────────────────────────────────
MVTEC_ROOT = None
for candidate in [
    REPO_ROOT / 'data' / 'mvtec',
    REPO_ROOT / 'data' / 'mvtec-ad',
    Path('/tmp/mvtec'),
]:
    if candidate.exists() and any(candidate.iterdir()):
        MVTEC_ROOT = candidate
        break

assert MVTEC_ROOT, f'MVTec not found. Expected at {REPO_ROOT}/data/mvtec'
categories = sorted([d.name for d in MVTEC_ROOT.iterdir() if d.is_dir()])
print(f'MVTec root : {MVTEC_ROOT}')
print(f'Categories : {len(categories)} → {categories}')

MVTec root : /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
Categories : 15 → ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


In [4]:
# ── Cell 4: Configure ───────────────────────────────────────────────────────
MODEL      = 'claude-opus-4-7'
PROMPT_KEY = 'manufacturing.detailed'
BATCH_SIZE = 10      # images per API call — max 20, 10 is reliable
BUDGET_USD = 5.0     # hard cap — raise after adding more credit

est_images = len(categories) * 83
est_cost   = est_images * 0.028   # ~$28 / 1M input tokens for Opus
print(f'Model        : {MODEL}')
print(f'Batch size   : {BATCH_SIZE} images/call')
print(f'Budget cap   : ${BUDGET_USD}  ← raise this when adding more credit')
print(f'Est. images  : ~{est_images}')
print(f'Est. cost    : ~${est_cost:.2f} full sweep')
print(f'  $5 covers  : ~{int(5 / (est_cost / est_images))} images (~{int(5 / (est_cost / est_images) / 83)} categories)')

Model        : claude-opus-4-7
Batch size   : 10 images/call
Budget cap   : $5.0  ← raise this when adding more credit
Est. images  : ~1245
Est. cost    : ~$34.86 full sweep
  $5 covers  : ~178 images (~2 categories)


In [5]:
# ── Cell 5: SMOKE TEST — 1 image, verify API before full sweep ──────────────
# (backend is built in Cell 6 — run that first, then optionally run this smoke test)

smoke_img = next((MVTEC_ROOT / 'bottle' / 'test').rglob('*.png'))
print(f'Smoke test image: {smoke_img.relative_to(REPO_ROOT)}')

smoke_result = backend.predict(
    smoke_img,
    'Is there any defect or anomaly? Reply with JSON only: '
    '{"is_anomalous": bool, "confidence": float, "defect_type": str, "description": str}'
)

print(f'  is_anomalous : {smoke_result.is_anomalous}')
print(f'  confidence   : {smoke_result.confidence:.2f}')
print(f'  defect_type  : {smoke_result.defect_type}')
print(f'  description  : {smoke_result.description[:100]}')
print(f'  latency_ms   : {smoke_result.latency_ms:.0f}')
print(f'  cost_usd     : ${smoke_result.cost_usd:.6f}')
print(f'  tokens_in    : {smoke_result.tokens_in}')
print(f'  tokens_out   : {smoke_result.tokens_out}')
print(f'  parse_error  : {smoke_result.parse_error}')
print()
print('✓ Smoke test passed — API key and model are working.')

Smoke test image: data/mvtec/bottle/test/broken_small/002.png


2026-05-24T02:38:32.532057Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-24T02:38:32.535246Z [info     ] anthropic.predict              [vlm_anomaly.backends.anthropic_backend] confidence=0.95 cost_usd=0.02425 image=002.png is_anomalous=True latency_ms=2994 model=claude-opus-4-7 parse_error=False tokens_in=1157 tokens_out=92


  is_anomalous : True
  confidence   : 0.95
  defect_type  : chip/contamination on rim
  description  : A visible chip or glass damage with residue is present on the right side of the bottle's inner rim, 
  latency_ms   : 2994
  cost_usd     : $0.024255
  tokens_in    : 1157
  tokens_out   : 92
  parse_error  : False

✓ Smoke test passed — API key and model are working.


In [5]:
# ── Cell 6: Build shared objects ────────────────────────────────────────────
from vlm_anomaly.backends.anthropic_backend import AnthropicBackend
from vlm_anomaly.config import Settings
from vlm_anomaly.datasets.mvtec import MVTec
from vlm_anomaly.evaluators.prompt_library import PromptLibrary
from vlm_anomaly.logging import configure_logging

configure_logging(log_level='INFO')
backend = AnthropicBackend(model=MODEL)

settings = Settings(
    _env_file=str(REPO_ROOT / '.env'),
    data_dir=str(MVTEC_ROOT.parent),
    results_dir=str(RESULTS_DIR),
    default_budget_usd=BUDGET_USD,
)
settings.results_dir = RESULTS_DIR

dataset    = MVTec(root_dir=MVTEC_ROOT)
prompt_lib = PromptLibrary(prompts_dir=PROMPTS_DIR)
prompt_str = prompt_lib.render(PROMPT_KEY)

print(f'Dataset  : {MVTEC_ROOT}')
print(f'Backend  : {backend.name} / {MODEL}')
print(f'Prompt   : {PROMPT_KEY} ({len(prompt_str)} chars)')

Dataset  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
Backend  : anthropic / claude-opus-4-7
Prompt   : manufacturing.detailed (587 chars)


In [6]:
# ── Cell 7: Run all categories (batched, idempotent, resumable) ─────────────
import json
import uuid as _uuid
from datetime import timezone, datetime
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, f1_score
from vlm_anomaly.logging import configure_logging

configure_logging(log_level='INFO')

MODEL_ID     = f'anthropic/{MODEL}'
total_cost   = 0.0
all_category_results = []


def _find_existing(results_dir, category, model_id):
    """Return (file, n_done) for a matching partial or complete run, else (None, 0)."""
    for f in results_dir.glob(f'*_mvtec_{category}.jsonl'):
        if f.stat().st_size < 10:
            continue
        try:
            lines = [l for l in f.read_text().splitlines() if l.strip()]
            if not lines:
                continue
            rec = json.loads(lines[0])
            if rec.get('model_id') == model_id:
                return f, len(lines)
        except Exception:
            pass
    return None, 0


for category in tqdm(categories, desc='MVTec categories'):
    all_samples = dataset.samples(category, split='test')
    total       = len(all_samples)

    existing_file, n_done = _find_existing(RESULTS_DIR, category, MODEL_ID)

    if n_done >= total:
        print(f'  [skip]   {category} — complete ({n_done}/{total})')
        continue
    elif n_done > 0:
        print(f'  [resume] {category} — continuing from image {n_done}/{total}')
        out_file       = existing_file
        samples_to_run = all_samples[n_done:]
    else:
        experiment_id  = _uuid.uuid4().hex[:8]
        out_file       = RESULTS_DIR / f'{experiment_id}_mvtec_{category}.jsonl'
        samples_to_run = all_samples

    images = [s.image_path for s in samples_to_run]
    labels = [s.label      for s in samples_to_run]
    rows   = []
    cat_cost = 0.0

    for i in tqdm(range(0, len(images), BATCH_SIZE), desc=category, leave=False):
        batch_imgs = images[i : i + BATCH_SIZE]
        results    = backend.predict_batch(batch_imgs, prompt_str)

        for pred, label in zip(results, labels[i : i + BATCH_SIZE]):
            row = {
                'experiment_id': out_file.stem.split('_')[0],
                'model_id':       MODEL_ID,
                'backend':        'anthropic',
                'dataset':        'mvtec',
                'category':       category,
                'sample_label':   label,
                'prediction': {
                    'image_path':   str(pred.image_path),
                    'is_anomalous': pred.is_anomalous,
                    'confidence':   pred.confidence,
                    'description':  pred.description,
                    'defect_type':  pred.defect_type,
                    'regions':      pred.regions,
                    'raw_response': pred.raw_response,
                    'latency_ms':   pred.latency_ms,
                    'cost_usd':     pred.cost_usd,
                    'tokens_in':    pred.tokens_in,
                    'tokens_out':   pred.tokens_out,
                    'parse_error':  pred.parse_error,
                },
                'timestamp': datetime.now(timezone.utc).isoformat(),
            }
            rows.append(row)
            with out_file.open('a') as fh:
                fh.write(json.dumps(row) + '\n')

        cat_cost += sum(r.cost_usd for r in results)

    if not rows:
        continue

    # Compute metrics over ALL rows in the file (including any resumed portion)
    all_lines = [json.loads(l) for l in out_file.read_text().splitlines() if l.strip()]
    gt     = [r['sample_label']               for r in all_lines]
    scores = [r['prediction']['confidence'] if r['prediction']['is_anomalous']
              else 1 - r['prediction']['confidence'] for r in all_lines]
    preds  = [int(r['prediction']['is_anomalous']) for r in all_lines]
    try:
        auroc = roc_auc_score(gt, scores)
    except Exception:
        auroc = float('nan')
    f1 = f1_score(gt, preds, zero_division=0)

    total_cost += cat_cost
    all_category_results.append({
        'category': category, 'auroc': auroc, 'f1': f1,
        'n': len(all_lines), 'cost': cat_cost,
    })
    print(f'  {category:12s}  AUROC={auroc:.3f}  F1={f1:.3f}  '
          f'n={len(all_lines)}  cost=${cat_cost:.4f}')

print(f'\nDone. Total cost this run: ${total_cost:.4f}')


MVTec categories:   0%|          | 0/15 [00:00<?, ?it/s]

  [skip]   bottle — complete (83/83)
  [resume] cable — continuing from image 130/150


cable:   0%|          | 0/2 [00:00<?, ?it/s]

2026-05-24T03:26:29.378850Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-24T03:26:29.382595Z [info     ] anthropic.predict_batch        [vlm_anomaly.backends.anthropic_backend] batch_size=10 cost_usd=0.26447 latency_ms=11830 model=claude-opus-4-7 parse_errors=0 tokens_in=13911 tokens_out=744
2026-05-24T03:26:29.383366Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] confidence=0.85 image=000.png is_anomalous=True parse_error=False
2026-05-24T03:26:29.384047Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] confidence=0.95 image=001.png is_anomalous=True parse_error=False
2026-05-24T03:26:29.385021Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] confidence=0.95 image=002.png is_anomalous=True parse_error=False
2026-05-24T03:26:29.385707Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] con

  cable         AUROC=0.940  F1=0.857  n=150  cost=$0.5240


...

2026-05-24T03:49:07.899633Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-24T03:49:07.902870Z [info     ] anthropic.predict_batch        [vlm_anomaly.backends.anthropic_backend] batch_size=10 cost_usd=0.25172 latency_ms=9620 model=claude-opus-4-7 parse_errors=0 tokens_in=13911 tokens_out=574
2026-05-24T03:49:07.903554Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] confidence=0.95 image=000.png is_anomalous=True parse_error=False
2026-05-24T03:49:07.904197Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] confidence=0.95 image=001.png is_anomalous=True parse_error=False
2026-05-24T03:49:07.905026Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] confidence=0.93 image=002.png is_anomalous=True parse_error=False
2026-05-24T03:49:07.905800Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] conf

  wood          AUROC=0.999  F1=0.976  n=79  cost=$2.0039


zipper:   0%|          | 0/16 [00:00<?, ?it/s]

2026-05-24T03:50:37.494373Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-24T03:50:37.499302Z [info     ] anthropic.predict_batch        [vlm_anomaly.backends.anthropic_backend] batch_size=10 cost_usd=0.26079 latency_ms=12021 model=claude-opus-4-7 parse_errors=0 tokens_in=13911 tokens_out=695
2026-05-24T03:50:37.500609Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] confidence=0.85 image=000.png is_anomalous=False parse_error=False
2026-05-24T03:50:37.501636Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] confidence=0.92 image=001.png is_anomalous=True parse_error=False
2026-05-24T03:50:37.503117Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] confidence=0.8 image=002.png is_anomalous=False parse_error=False
2026-05-24T03:50:37.504606Z [info     ] anthropic.predict_batch.item   [vlm_anomaly.backends.anthropic_backend] co

  zipper        AUROC=0.873  F1=0.849  n=151  cost=$3.8712

Done. Total cost this run: $34.9203


In [7]:
# ── Cell 7b: Leaderboard v2─────────────────────────────────────────────────────
import importlib
import vlm_anomaly.analysis.aggregator as _agg_mod
importlib.reload(_agg_mod)
from vlm_anomaly.analysis.aggregator import leaderboard, cost_accuracy_table

lb = leaderboard(RESULTS_DIR)
if lb.empty:
    print('No results yet.')
else:
    summary = cost_accuracy_table(RESULTS_DIR)
    print('=== Summary (mean across categories) ===')
    print(summary[["model_id","mean_auroc","mean_latency_ms"]].to_string(index=False))
    print()
    print('=== Per-category breakdown (all models) ===')
    clean = lb[lb["model_id"].notna() & lb["category"].notna() & (lb["n_images"] > 10)]
    display(
        clean[["model_id","category","n_images","auroc","f1","mean_latency_ms"]]
        .sort_values(["model_id","auroc"], ascending=[True,False])
        .reset_index(drop=True)
    )

/Users/sabareeswarans/Projects_26/VLM-Anomaly/src/vlm_anomaly/analysis/aggregator.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(frames, ignore_index=True)
/Users/sabareeswarans/Projects_26/VLM-Anomaly/src/vlm_anomaly/analysis/aggregator.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(frames, ignore_index=True)


=== Summary (mean across categories) ===
                             model_id  mean_auroc  mean_latency_ms
            anthropic/claude-opus-4-7    0.751931      1080.101496
              gemini/gemini-2.5-flash    0.610432      3971.451703
openrouter/qwen/qwen3-vl-32b-instruct    0.178223      2122.039152
         claude_cli/claude-sonnet-4-6         NaN      7117.170888

=== Per-category breakdown (all models) ===


,model_id,category,n_images,auroc,f1,mean_latency_ms
0,anthropic/claude-opus-4-7,bottle,83,0.910317,0.854839,1030.439475
1,anthropic/claude-opus-4-7,tile,117,0.908009,0.958084,1045.406576
2,anthropic/claude-opus-4-7,wood,79,0.890351,0.975610,1100.915710
3,anthropic/claude-opus-4-7,cable,150,0.885026,0.857143,1206.414767
4,anthropic/claude-opus-4-7,hazelnut,110,0.870893,0.832117,1046.069879
5,anthropic/claude-opus-4-7,toothbrush,42,0.851389,0.848485,1175.806569
6,anthropic/claude-opus-4-7,metal_nut,115,0.826246,0.816092,954.926203
7,anthropic/claude-opus-4-7,grid,78,0.796157,0.982456,1091.513829
8,anthropic/claude-opus-4-7,leather,124,0.706861,0.994595,989.908377
9,anthropic/claude-opus-4-7,transistor,100,0.698958,0.589474,1110.370656


In [8]:
# ── Cell 8: Summary table ────────────────────────────────────────────────────
import pandas as pd

if all_category_results:
    df = pd.DataFrame(all_category_results)
    print(f'Mean AUROC : {df.auroc.mean():.3f}')
    print(f'Mean F1    : {df.f1.mean():.3f}')
    print(f'Total cost : ${df.cost.sum():.4f}')
    print()
    display(df.sort_values('auroc', ascending=False).reset_index(drop=True))
else:
    print('No results yet — run Cell 7 first.')

Mean AUROC : 0.885
Mean F1    : 0.870
Total cost : $34.9203



,category,auroc,f1,n,cost
0,leather,1.000000,0.994595,124,3.119505
1,wood,0.999123,0.975610,79,2.003880
2,carpet,0.995987,0.972067,117,2.974335
3,tile,0.995310,0.958084,117,2.158815
4,grid,0.993734,0.982456,78,1.993215
5,cable,0.939561,0.857143,150,0.523980
6,pill,0.894299,0.900763,167,2.950395
7,hazelnut,0.890536,0.832117,110,2.796615
8,zipper,0.872768,0.849057,151,3.871230
9,toothbrush,0.872222,0.848485,42,1.083480


In [9]:
# ── Cell 8: Generate report ──────────────────────────────────────────────────
from vlm_anomaly.analysis.report_generator import generate

report = generate(RESULTS_DIR, str(REPO_ROOT / 'REPORT.md'))
print(f'Report written to {report}')

2026-05-24T03:56:24.807550Z [info     ] report.generate.start          [vlm_anomaly.analysis.report_generator] results_dir=/Users/sabareeswarans/Projects_26/VLM-Anomaly/results
/Users/sabareeswarans/Projects_26/VLM-Anomaly/src/vlm_anomaly/analysis/aggregator.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(frames, ignore_index=True)
/Users/sabareeswarans/Projects_26/VLM-Anomaly/src/vlm_anomaly/analysis/aggregator.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combi

Report written to /Users/sabareeswarans/Projects_26/VLM-Anomaly/REPORT.md
